# 019 熵交叉熵与KL散度

## 目标
从有限概率向量逐项计算H、交叉熵与两个方向的KL，检查零支持，并解释受限候选选择。全部数据为原创合成；数学背景见lecture.pdf。

执行结果：主例为1.5、1.75、0.25 bit；支持集错误的正向KL为无穷；受限集合正向选宽，反向左右并列。重启内核后从头运行。

## 设置
核心计算只依赖Python标准库。当前工作目录应为本单元。Notebook额外需要IPython/ipykernel，读取预制图片无需重新绘图。

In [1]:
from pathlib import Path
import math, json, sys
from fractions import Fraction
from decimal import Decimal, localcontext
from experiment import (load_distributions, load_labels, entropy, cross_entropy, kl, kl_terms,
                        empirical_distribution, mean_nll, mix_with_uniform, run)
print("Python", sys.version.split()[0])
print("全为本地合成数据，无网络或付费API")

Python 3.12.14
全为本地合成数据，无网络或付费API


## 步骤
### 1 检查有序分布
A、B、C的列顺序是概率的语义；长度相同还不够。

In [2]:
ds = load_distributions()
p, q = ds["main_p"], ds["main_q"]
print("状态顺序 A, B, C")
print("P", p, "Q", q)
print("总和", math.fsum(p), math.fsum(q))

状态顺序 A, B, C
P (0.5, 0.25, 0.25) Q (0.25, 0.25, 0.5)
总和 1.0 1.0


### 2 逐项手算bit
先猜每一个贡献。KL的负分项不能裁成0。

In [3]:
h_bits = entropy(p, 2)
ce_bits = cross_entropy(p, q, 2)
d_bits = kl(p, q, 2)
print("H, CE, KL (bit):", h_bits, ce_bits, d_bits)
print("KL各项:", kl_terms(p, q, 2))
assert (h_bits, ce_bits, d_bits) == (1.5, 1.75, .25)

H, CE, KL (bit): 1.5 1.75 0.25
KL各项: (0.5, 0.0, -0.25)


### 3 转换单位并检查分解
独立手算提供参考；恒等式本身不是独立正确性证明。

In [4]:
h, ce, d = entropy(p), cross_entropy(p, q), kl(p, q)
print("H, CE, KL (nat):", h, ce, d)
print("CE − H − KL:", ce-h-d)
assert abs(ce-h-d) < 1e-14
assert abs(h-h_bits*math.log(2)) < 1e-14

H, CE, KL (nat): 1.0397207708399179 1.2130075659799042 0.17328679513998632
CE − H − KL: 2.7755575615628914e-17


### 4 独立高精度参考
从精确有理数构造Decimal，使用80位ln，不调用核心函数内部的math.log。

In [5]:
with localcontext() as context:
    context.prec = 80
    pd = [Decimal(1)/2, Decimal(1)/4, Decimal(1)/4]
    qd = [Decimal(1)/4, Decimal(1)/4, Decimal(1)/2]
    ref_ce = -sum(a*b.ln() for a,b in zip(pd, qd))
    ref_kl = sum(a*(a.ln()-b.ln()) for a,b in zip(pd, qd))
print("Decimal CE:", ref_ce)
print("Decimal KL:", ref_kl)
assert abs(float(ref_ce)-ce) < 1e-14
assert abs(float(ref_kl)-d) < 1e-14

Decimal CE: 1.2130075659799042914801562125518089941321252351304466947111900166134388384469657
Decimal KL: 0.17328679513998632735430803036454414201887503359006381353017000237334840549242367


### 5 零支持与精确无穷
共同零项跳过。P正质量遇到Q零质量才产生正向无穷。

In [6]:
p0, q0 = ds["zero_p"], ds["zero_q"]
print("交叉熵:", cross_entropy(p0,q0))
print("正反KL:", kl(p0,q0), kl(q0,p0))
print("共同零:", kl([1,0],[1,0]))
assert math.isinf(kl(p0,q0))
assert abs(kl(q0,p0)-math.log(2)) < 1e-14
assert kl([1,0],[1,0]) == 0

交叉熵: inf
正反KL: inf 0.6931471805599453
共同零: 0.0


### 6 极小正概率仍有有限损失
不要把支持集上的零和小正数混淆。下式第二个向量只在浮点精度内归一化；代码接受舍入级误差。

In [7]:
print("1e-300对应损失:", cross_entropy([0,1],[1,1e-300]))
for power in [1,3,6,12]:
    epsilon=10.0**(-power)
    q_epsilon=[1-epsilon,epsilon]
    print(power, "正向",kl([.5,.5],q_epsilon), "反向",kl(q_epsilon,[.5,.5]))

1e-300对应损失: 690.7755278982137
1 正向 0.5108256237659905 反向 0.36806420716849714
3 正向 2.761230709097915 反向 0.6852399254477132
6 正向 6.2146085984224415 反向 0.6931323650498873
12 正向 13.12236337740483 反向 0.6931471805313143


### 7 检验非对称性
主例恰好对称不是定理。这一对二分类分布两个方向不同。

In [8]:
pa,qa=[.75,.25],[.5,.5]
print("D(P||Q):",kl(pa,qa))
print("D(Q||P):",kl(qa,pa))
assert abs(kl(pa,qa)-kl(qa,pa)) > .01

D(P||Q): 0.13081203594113697
D(Q||P): 0.14384103622589045


### 8 只在三个候选中选择
所有候选严格正。差异来自权重和受限集合，不依赖强行设置0。

In [9]:
directional=[]
for name in ["broad","left","right"]:
    forward=kl(ds["target"],ds[name])
    reverse=kl(ds[name],ds["target"])
    directional.append((name,forward,reverse))
    print(name, "正向", forward, "反向", reverse)
print("加入目标本身:",kl(ds["target"],ds["target"]))
assert directional[0][1] < directional[1][1]
assert directional[1][2] < directional[0][2]

broad 正向 0.32128893843963136 反向 0.6809619717262487
left 正向 1.237811159854801 反向 0.5816533152719076
right 正向 1.237811159854801 反向 0.5816533152719076
加入目标本身: 0.0


### 9 经验分布与逐样本平均
这里同一个Q用于全部观察；普通条件分类器需在每个输入上使用对应Q。

In [10]:
labels=load_labels()
p_hat=empirical_distribution(labels,3)
print("labels",labels,"P_hat",p_hat)
print("逐样本平均",mean_nll(labels,q))
print("经验交叉熵",cross_entropy(p_hat,q))
assert abs(mean_nll(labels,q)-cross_entropy(p_hat,q))<1e-14

labels [0, 0, 1, 2] P_hat (0.5, 0.25, 0.25)
逐样本平均 1.2130075659799042
经验交叉熵 1.2130075659799042


### 10 有意触发拒绝
合法支持集错误返回无穷；非法概率向量应拒绝。

In [11]:
for bad in ([.2,.2],[-.1,1.1],[True,0],[math.nan,1]):
    try:
        entropy(bad)
    except ValueError as exc:
        print("预期拒绝:",str(exc))
    else:
        raise AssertionError("invalid distribution was accepted")

预期拒绝: probability total must be 1 within 1e-12
预期拒绝: every probability must be in [0,1]
预期拒绝: probabilities must be finite real numbers, not booleans
预期拒绝: nonfinite or underflowed probability


### 11 平滑确实改变了模型
与均匀分布混合保持归一化。不能把新损失当原模型的另一个实现值。

In [12]:
smoothed=mix_with_uniform(q0,.02)
print("新Q",smoothed)
print("新交叉熵",cross_entropy(p0,smoothed))
print("原交叉熵",cross_entropy(p0,q0))
assert all(v>0 for v in smoothed)
assert abs(cross_entropy(p0,smoothed)-2.512029157214198)<1e-14
print("100次未见罕见状态概率:",.999**100)

新Q (0.9866666666666667, 0.006666666666666667, 0.006666666666666667)
新交叉熵 2.512029157214198
原交叉熵 inf
100次未见罕见状态概率: 0.9047921471137089


## 检查
### 12 生成并对齐脚本结果
Notebook写入notebook_outputs，和已随附的脚本outputs逐字节比较。两个路径分开，避免覆盖参考。

In [13]:
report=run(Path("notebook_outputs"))
for filename in ["summary.json","directional.csv","epsilon.csv"]:
    assert (Path("notebook_outputs")/filename).read_bytes() == (Path("outputs")/filename).read_bytes()
    print(filename,"与脚本结果一致")
print(report["best_candidates"])

summary.json 与脚本结果一致
directional.csv 与脚本结果一致
epsilon.csv 与脚本结果一致
{'forward': ['broad'], 'reverse': ['left', 'right']}


### 13 独立测试套件
涵盖784个有理分布对、363条短序列、拒绝输入与新进程复现。

In [14]:
import unittest
import test_experiment
suite=unittest.defaultTestLoader.loadTestsFromModule(test_experiment)
result=unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()

.

.

.

.

.

...

.

.

.

.

----------------------------------------------------------------------
Ran 11 tests in 1.490s

OK


## 下一步
1. 从零重写支持集检查，再与参考函数比较。
2. 把目标P加入受限集合，解释两个方向都选P的意义。
3. 阅读answers.pdf，完成经验0但总体无穷的解释。

已验证范围：新进程中的真实InProcessKernel顺序执行、脚本结果一致。未测试全新Anaconda安装、浏览器Notebook前端、socket传输或跨平台环境。所有合成结果只用于机制理解。